In [ ]:
# Allow importing from src
import sys
sys.path.insert(0, '../src/')

# Fix for draw_geometries crashing on Wayland
import os
os.environ["XDG_SESSION_TYPE"] = "x11"

In [ ]:
import open3d as o3d
from pathlib import Path
import numpy as np
import torch

import utils as U

In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import argparse
from ingp import LInstantNGP
from pathlib import Path
import re
import open3d as o3d
import utils as U


COMPUTE_DEVICE = torch.device('cpu')
if torch.cuda.is_available():
    COMPUTE_DEVICE = torch.device('cuda:0')
elif torch.mps.is_available():
    COMPUTE_DEVICE = torch.device('mps')
print(f"{COMPUTE_DEVICE=}")


def get_origin_direction_eq_angles(n, img_shape, focal):
    phis, thetas = U.rays.equidistance_rotations(n)

    origin, direction = [], []
    for phi, theta in zip(phis, thetas):
        o, d = U.rays.create_rays(
            img_shape[0],
            img_shape[1],
            U.rays.create_intrinsic(focal, img_shape),
            U.rays.look_at(4, phi, theta)
        )
        origin.append(o)
        direction.append(d)

    return torch.stack(origin), torch.stack(direction)


def get_origin_direction_c2w_intrinsic(img_shape, c2ws, intrinsics):
    origin, direction = [], []
    for c2w, intrinsic in zip(c2ws, intrinsics):
        o, d = U.rays.create_rays(img_shape[0], img_shape[1], intrinsic, c2w)
        origin.append(o)
        direction.append(d)

    return torch.stack(origin), torch.stack(direction)


def load_model_and_data(log_path):
    hparams_path = log_path / "hparams.yaml"

    chkpts = list((log_path / "checkpoints").glob("*best_val*"))
    chkpts.sort(key=lambda p: int(re.match(r".*epoch=(\d*).*", p.name, flags=re.DOTALL).group(1)))
    chkpt_path = chkpts[-1]
    print(f"Using checkpoint: {chkpt_path}")

    model = LInstantNGP.load_from_checkpoint(
        chkpt_path, map_location=COMPUTE_DEVICE, hparams_file=hparams_path
    )
    model.freeze()
    model.eval()

    data = U.lutils.NeRFData.load_from_checkpoint(
        chkpt_path, map_location=torch.device('cpu'), hparams_file=hparams_path
    )
    data._set_hparams(model.hparams)
    data.setup("predict")

    return model, data


def cloud_from_tensor(tens):
    return o3d.geometry.PointCloud(o3d.utility.Vector3dVector(tens.cpu()))


proj_dir = Path(f"..").resolve()
if not proj_dir.exists():
    raise ValueError(f"Specified logs at {proj_dir} don't exist")
log_path = (proj_dir / "lightning_logs" / "ingp_DTU_scan114_m_ds").resolve()

model, data = load_model_and_data(log_path)
datadir = (proj_dir / "data" / data.hparams.source / data.hparams.name).resolve()

idxs = U.data.find_mn_angles(data.c2ws, angle_count=8)
origin, direction = get_origin_direction_c2w_intrinsic(
    (data.images.shape[1], data.images.shape[2]),
    data.c2ws[idxs], data.intrinsics[idxs]
)
alpha_mask = data.images[idxs, ..., -1] != 0
origin, direction = origin[alpha_mask], direction[alpha_mask]
dl = DataLoader(TensorDataset(origin, direction), batch_size=2**9)

print(f"Running Surface Point extraction for {dl.dataset.tensors[0].shape[0]:_d} rays")
dm_depth = []
with torch.no_grad():
    for o, di in tqdm(dl, total=len(dl), unit="batch", postfix="batch_size=2^9"):
        o, di = o.to(model.device), di.to(model.device)
        _, de, acc, _ = model.render_rays(o, di)
        points = o + de * di
        mask = model.nerf(points, None, only_sigma=True) < model.hparams.f_sigma_threshold

        near_plane = torch.sqrt(torch.sum(torch.pow(o, 2), -1, keepdim=True)) + model.near_offset
        de[mask | (acc < 0.99) | (de < near_plane)] = torch.inf

        dm_depth.append(de.cpu())
    dm_depth = torch.cat(dm_depth, 0)

#torch.save(dm_depth, "temp.pt")
#dm_depth = torch.load("temp.pt")

mask = (dm_depth != torch.inf).squeeze(-1)
adjustment = (model.far_offset - model.near_offset) / 1024 / 2
dm_points = origin[mask] + (dm_depth[mask] - adjustment) * direction[mask]
bbox_mask = (dm_points.abs() <= 1.0).all(-1)
dm_points = dm_points[bbox_mask]
print(f"Points within [-1, 1] bbox limits: {dm_points.shape[0]:_d}")

point_cloud = cloud_from_tensor(dm_points)

print(f"Calculating normal vectors...")
normals = []
normal_points = DataLoader(
    torch.from_numpy(np.asarray(point_cloud.points, dtype=np.float32)),
    batch_size=2**19, shuffle=False
)
for nps in normal_points:
    normals.append(model.estimate_normals(nps).cpu())
normals = torch.cat(normals, dim=0)
point_cloud.normals = o3d.utility.Vector3dVector(normals)

point_cloud = point_cloud.voxel_down_sample(0.002)
points = torch.from_numpy(np.asarray(point_cloud.points, dtype=np.float32)).to(model.device)
normals = torch.from_numpy(np.asarray(point_cloud.normals, dtype=np.float32)).to(model.device)
points.shape

In [ ]:
import torch
import numpy as np

def project_points_to_camera(points: torch.Tensor, c2w: torch.Tensor, intrinsic: torch.Tensor, 
                             image_size: tuple[int, int] = (800, 800)
                             ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Project 3D points to camera coordinates and check visibility.
    
    Args:
        points (shape[N, 3]): Tensor of 3D points
        c2w (shape[4, 4]): Extrinisic camera matrix (Camera to World)
        intrinsic (shape[3, 3]): Intrinsic camera matrix
        image_size (height, width): Image dimensions
        
    Returns:
        uv (shape[N, 2)]): UV coordinates in image space
        depth (shape[N]): Depth values in world
        valid_mask (shape[N]): Boolean mask of points within image bounds
    """
    device = points.device
    N = points.shape[0]
    height, width = image_size

    # Defining depth in world space to make it more universal for Hash Encoding as it limits to a bbox here
    depth = torch.sqrt(torch.sum((points - c2w[:3, -1]) ** 2, dim=-1))
    
    # Utilizing projection to image space to find valid points
    points_homo = torch.cat([points, torch.ones(N, 1, device=device)], dim=1)
    w2c = torch.linalg.inv(c2w)
    points_cam = (w2c @ points_homo.T).T
    points_cam_3d = points_cam[:, :3]
    points_image = (intrinsic @ points_cam_3d.T).T
    
    # Points are valid if within image bounds and in front of cam
    uv = points_image[:, :2] / points_image[:, 2:3]
    u, v = uv[:, 0], uv[:, 1]
    valid_u = (u >= 0) & (u <= width)
    valid_v = (v >= 0) & (v <= height)
    # Using camera space depth is simpler here as world space depth doesn't have a sign
    valid_depth = points_cam[:, 2] < 0
    valid_mask = valid_u & valid_v & valid_depth
    
    return uv, depth, valid_mask


def calculate_point_visibility(points: torch.Tensor, normals: torch.Tensor, c2ws: torch.Tensor,
                               intrinsics: torch.Tensor, image_size: tuple[int, int] = (800, 800),
                               pixel_scaler: int = 1, depth_tolerance: float = 0.01) -> torch.Tensor:
    """
    Calculate visibility count for each point from multiple cameras.
    Points are visible if they are within depth_tolerance of the closest point for their pixel.
    
    Args:
        points (shape[N, 3]): Tensor of 3D points
        normals (shape[N, 3]): Normals for points
        c2ws (shape[M, 4, 4]): Extrinisic camera matrices (Camera to World)
        intrinsics (shape[M, 3, 3]): Intrinsic camera matrices
        image_size (height, width): Image dimensions
        pixel_scaler: pixels are considered as this size (1 is original, e.g. 2 makes width, height act like halved)
        depth_tolerance: points within this depth range of the closest point are considered visible
        
    Returns:
        visibility_count (shape[N]): Tensor with visibility count for each point
    """
    device = points.device
    N = points.shape[0]
    height, width = image_size
    
    visibility_count = torch.zeros(N, dtype=torch.int32, device=device)
    back_facing = torch.zeros(N, dtype=torch.int32, device=device)
    
    for c2w, intrinsic in zip(c2ws, intrinsics):
         # Usable for normal orientation
        to_cam_direction = torch.nn.functional.normalize(c2w[None, :3, -1] - points, p=2, dim=-1)
                                                         
        uv, depth, valid_mask = project_points_to_camera(points, c2w, intrinsic, image_size)
        
        valid_indices = torch.where(valid_mask)[0]
        if len(valid_indices) == 0:
            continue
            
        valid_uv = uv[valid_indices]
        valid_depth = depth[valid_indices]
        
        # Finding closest pixel to point
        u_pixel = torch.round(valid_uv[:, 0]).long() // pixel_scaler * pixel_scaler
        v_pixel = torch.round(valid_uv[:, 1]).long() // pixel_scaler * pixel_scaler
        
        # Create pixel index tensor for grouping
        pixel_indices = v_pixel * width + u_pixel
        unique_pixels, inverse_indices = torch.unique(pixel_indices, return_inverse=True)
        
        # Closest points by depth per pixel
        min_depths = torch.zeros_like(unique_pixels, dtype=torch.float32)
        for i, pixel_idx in enumerate(unique_pixels):
            mask = (pixel_indices == pixel_idx)
            min_depths[i] = valid_depth[mask].min()
        
        # Keeping points within tolerance to depth
        point_min_depths = min_depths[inverse_indices]
        depth_differences = valid_depth - point_min_depths
        within_tolerance = depth_differences <= depth_tolerance
        
        # Mark visible points
        visible_indices = valid_indices[within_tolerance]
        visibility_count[visible_indices] += 1

        # If to_cam and normal faces opposite directions, we see the "back", usually an artifact of cloud generation
        dots = torch.sum(to_cam_direction[visible_indices] * normals[visible_indices], dim=-1)
        # Checking degenerate normals with the second term here
        back_facing[visible_indices] += (dots < 0) | (normals[visible_indices] == 0).all(-1)
        
    return visibility_count, back_facing


def get_visibility_mask(points: torch.Tensor, normals: torch.Tensor, c2ws: list[torch.Tensor],
                        intrinsics: list[torch.Tensor], image_size: tuple[int, int], pixel_scaler: int = 1,
                        depth_tolerance: float = 0.01, min_visibility: int = 1, max_back_face: int = 0
                        ) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Complete pipeline for processing NeRF point cloud with visibility filtering.
    
    Args:
        points (shape[N, 3]): Tensor of 3D points
        normals (shape[N, 3]): Normals for points
        c2ws (shape[M, 4, 4]): Extrinisic camera matrices (Camera to World)
        intrinsics (shape[M, 3, 3]): Intrinsic camera matrices
        image_size (height, width): Image dimensions
        pixel_scaler: pixels are considered as this size (1 is original, e.g. 2 makes width, height act like halved)
        depth_tolerance: points within this depth range of the closest point are considered visible
        min_visibility: minimum cameras that must see a point to keep it
        max_back_face: maximum allowed angles from which the point normal faces away from
        
    Returns:
        tuple: tuple containing (visibility_mask, back_facing_mask)
            - **visibility_mask**: *shape[N]*: Visibility mask of points above threshold
            - **back_facing_mask**: *shape[N]*: Back facing mask of points below threshold
    """
    visibility_count, back_facing = calculate_point_visibility(
        points, normals, c2ws, intrinsics, image_size, pixel_scaler, depth_tolerance
    )

    visibility_mask = visibility_count >= min_visibility
    back_facing_mask = back_facing > max_back_face
    return visibility_mask, back_facing_mask

# Process point cloud
vis_mask, bf_mask = get_visibility_mask(
    points, normals, data.c2ws[idxs].to(model.device), data.intrinsics[idxs].to(model.device), 
    image_size=tuple(data.images.shape[1:3]),
    pixel_scaler=4,
    depth_tolerance=0.002,
    min_visibility=1,
    max_back_face=0,
)

vis_mask.sum().item(), bf_mask.sum().item()

In [ ]:
torch.isclose(normals, torch.zeros_like(normals), atol=1e-5).all(-1).sum()

In [ ]:
p, n = points.clone(), normals.clone()
print(len(p))

vis_mask, bf_mask = get_visibility_mask(
    p, n, data.c2ws[idxs].to(model.device), data.intrinsics[idxs].to(model.device), 
    image_size=tuple(data.images.shape[1:3]),
    pixel_scaler=4,
    depth_tolerance=0.002,
    min_visibility=1,
    max_back_face=0,
)
mask = vis_mask & (~bf_mask)
p, n = p[mask], n[mask]
print(len(p), vis_mask.sum().item(), bf_mask.sum().item())

In [ ]:
pc = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(p.cpu().numpy()))
pc.normals = o3d.utility.Vector3dVector(n.cpu().numpy())
pc.paint_uniform_color([0.5,0.5,0.5])
o3d.visualization.draw_geometries([pc])

In [ ]:
pc

In [ ]:
vis_mask.sum(), bf_mask.sum()

In [ ]:
f = cloud_from_tensor(points)
colors = np.zeros(points.shape)

colors[vis_mask.cpu().numpy(), 0] = 1
colors[~vis_mask.cpu().numpy(), 1] = 1

colors[bf_mask.cpu().numpy(), 2] = 1

f.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([f])

In [ ]:
U.data.ObjectSource("DTU") is U.data.ObjectSource.DTU

In [ ]:
mesh, _ =o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pc, depth=9)
mesh.compute_vertex_normals()
mesh = mesh.filter_smooth_laplacian(5)
mesh.compute_vertex_normals()

o3d.visualization.draw_geometries([mesh])

In [ ]:
i, c, a = mesh.cluster_connected_triangles()
i, c, a = np.asarray(i), np.asarray(c), np.asarray(a)

mesh.remove_triangles_by_mask(np.isin(i, np.where(c < 1000)))

# Remove unused vertices and tidy up
mesh.remove_unreferenced_vertices()
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()

o3d.visualization.draw_geometries([mesh])